One useful library for viewing a topic model is [LDAvis](https://github.com/cpsievert/LDAvis), an R package for creating interactive web visualizations of topic models, and its Python port, [PyLDAvis](https://github.com/bmabey/pyLDAvis). This library is focused on visualizing a topic model, using PCA to chart the relationship between topics and between topics and words in the topic model. It is also agnostic about library you use to create the topic model, so long as you extract the necessary data in the correct formats.

While the python version of the library works very smoothly with Gensim, which I have discussed before, there is little documentation for how to move from a topic model created using MALLET to data that can be processed by the LDAvis library. For reasons that require their own blog post, I have shifted from using Gensim for my topic model to using MALLET (spoilers: better documentation of output formats, more widespread use in the humanities so better documentation and code examples generally). But I still wanted to use this library to visualize the full model as a way of generating an overall view of the relationship between the 250 topics it contains.

The documentation for both LDAvis and PyLDAvis relies primarily on code examples to demonstrate how to use the libraries. My primary sources were a [python example](http://nbviewer.jupyter.org/github/bmabey/pyLDAvis/blob/master/notebooks/pyLDAvis_overview.ipynb) and two R examples, [one focused on manipulating the model data](https://ldavis.cpsievert.me/newsgroup/newsgroup.html) and one on the [full model to visualization process](https://ldavis.cpsievert.me/reviews/reviews.html). The ["details" documentation](https://cran.r-project.org/web/packages/LDAvis/vignettes/details.pdf) for the R library also proved key for trouble-shooting when the outputs did not match my expectations. (Pro tip: word order matters.)

Looking at the examples, the data required for the visualization library are:

- topic-term distributions (matrix, `phi`)
- document-topic distributions (matrix, `theta`)
- document lengths (number vector)
- vocab (character vector)
- term frequencies (number vector)

One challenge is that the order of the data needs to be managed, so that the terms columns in `phi`, the topic-term matrix, are in the same order as the vocab vector, which is in the same order as the frequencies vector, and the documents index of `theta`, the document-topic matrix, is in the same order of the document lengths vector.

I was able to isolate and generate all of the data necessary for creating the visualization from the statefile produced by MALLET. For the document lengths, vocabulary, and frequencies, I computed the data on the tokens remaining after stopword removal, so only those tokens considered in the creation of the model. I used the optimize interval in creating the model to allow some topics to be more prominent than others. As a result, the `alpha` value is a list of values (weights per topic).

The first step was to extract the data from the MALLET statefile and into a pandas dataframe. 

In [3]:
import gzip
import os
import pandas as pd

dataDir = "/Users/jeriwieringa/Dissertation/data/model_outputs/"

def extract_params(statefile):
    """Extract the alpha and beta values from the statefile.

    Args:
        statefile (str): Path to statefile produced by MALLET.
    Returns:
        tuple: alpha (list), beta    
    """
    with gzip.open(statefile, 'r') as state:
        params = [x.decode('utf8').strip() for x in state.readlines()[1:3]]
    return (list(params[0].split(":")[1].split(" ")), float(params[1].split(":")[1]))


def state_to_df(statefile):
    """Transform state file into pandas dataframe.
    The MALLET statefile is tab-separated, and the first two rows contain the alpha and beta hypterparamters.
    
    Args:
        statefile (str): Path to statefile produced by MALLET.
    Returns:
        datframe: topic assignment for each token in each document of the model
    """
    return pd.read_csv(statefile,
                       compression='gzip',
                       sep=' ',
                       skiprows=[1,2]
                       )

In [4]:
params = extract_params(os.path.join(dataDir, 'target_300_10.18497.state.gz'))

In [5]:
alpha = [float(x) for x in params[0][1:]]
beta = params[1]
print("{}, {}".format(alpha, beta))

[0.029812924118304992, 0.009271861921224648, 0.036602792963554454, 0.03036346204521996, 0.020332718571530326, 0.10351890596388356, 0.012755495950966892, 0.04290588069973869, 0.01901706336256544, 0.010000562544823921, 0.008051737793126377, 0.010478767490879254, 0.05102267459262269, 0.017735208419080477, 0.040884777680179944, 0.02240422753511921, 0.04633390719823591, 0.009648564936291307, 0.01221798611153162, 0.00518484471239066, 0.026017510181194876, 0.012790226620751563, 0.01623316505431329, 0.020333139411403224, 0.00825133910195449, 0.009324396700901893, 0.009969352687469672, 0.014058852097669477, 0.014327024486392018, 0.007030687242074, 0.013643807447897123, 0.014522948963225387, 0.04471494487932859, 0.014164304504351164, 0.0039162429342773745, 0.014532343795810298, 0.025680848757922235, 0.01983317219162416, 0.03957426890109219, 0.032049504012664, 0.023936849755631102, 0.020045882246671455, 0.0163365826757628, 0.003664273571444934, 0.01244363813012928, 0.006379376114561163, 0.0318383

In [6]:
df = state_to_df(os.path.join(dataDir, 'target_300_10.18497.state.gz'))

One small complication I found with my data was there is a token `nan` in the list of "types." Because `nan` is used in Pandas to indicate missing integer values, Pandas assumed it was an integer, rather than a string. To address this misidentification problem, I explicitly set the type for the entire column "type" to string. (Two meanings of type. Also confusing.)

In [7]:
df['type'] = df.type.astype(str)
df[:10]

,#doc,source,pos,typeindex,type,topic
0,0,NaN,0,0,uncommon,165
1,0,NaN,1,1,thing,224
2,0,NaN,2,2,member,224
3,0,NaN,3,3,boarding,136
4,0,NaN,4,4,school,14
5,0,NaN,5,5,form,237
6,0,NaN,6,6,clique,94
7,0,NaN,7,7,pair,11
8,0,NaN,8,8,tend,136
9,0,NaN,9,9,thank,224


Above is a preview of the data that I have from the statefile, and which I am going to use generate the data needed for the LDAvis library. It contains the document id, the word position index, the word index, the word, and its topic assignment.

The first bit of data to gather is the length of the documents. To do this, I grouped the data by the document id and counted the tokens in the doc. This data is sorted by the document id, so is in the correct order for the visualization preprocessing.

In [8]:
# Get document lengths from statefile
docs = df.groupby('#doc')['type'].count().reset_index(name ='doc_length')

docs[:10]

,#doc,doc_length
0,0,116
1,1,102
2,2,108
3,3,97
4,4,121
5,5,113
6,6,123
7,7,137
8,8,121
9,9,132


The second bit of data is the vocabulary and frequencies. Here I used pandas to generate a new frame with the counts for each word. I then sorted this dataframe so that it is alphabetical by type, a step I will repeat in creating the topic-term matrix.

In [9]:
# Get vocab and term frequencies from statefile
vocab = df['type'].value_counts().reset_index()
vocab.columns = ['type', 'term_freq']
vocab = vocab.sort_values(by='type', ascending=True)

vocab[:10]

,type,term_freq
31917,aaa,77
54149,aaaa,18
48842,aaaaa,26
22602,aad,180
50363,aai,24
52811,aal,20
31326,aalborg,80
48847,aan,26
45025,aand,32
24653,aar,146


Clearly some OCR problems have managed to make their way through all of my preprocessing efforts. With the low term frequencies, however, they should be only minimally present in the final representation of the topic model.

The third step was to create the matrix files. Here is where things get a bit tricky, as there is the adding of smoothing values and normalizing the data so that each row sums to 1. To do the normalizing, I am used `sklearn` because these are large matrices that require a more optimized function than dividing by the sum of the row with pandas.

In [10]:
# Topic-term matrix from state file
# https://ldavis.cpsievert.me/reviews/reviews.html

import sklearn.preprocessing

def pivot_and_smooth(df, smooth_value, rows_variable, cols_variable, values_variable):
    """
    Turns the pandas dataframe into a data matrix.
    Args:
        df (dataframe): aggregated dataframe 
        smooth_value (float): value to add to the matrix to account for the priors
        rows_variable (str): name of dataframe column to use as the rows in the matrix
        cols_variable (str): name of dataframe column to use as the columns in the matrix
        values_variable(str): name of the dataframe column to use as the values in the matrix
    Returns:
        dataframe: pandas matrix that has been normalized on the rows.
    """
    matrix = df.pivot(index=rows_variable, columns=cols_variable, values=values_variable).fillna(value=0)
    matrix = matrix.values + smooth_value
    
    normed = sklearn.preprocessing.normalize(matrix, norm='l1', axis=1)
    
    return pd.DataFrame(normed)

First, we need to aggregate the data from the statefile dataframe to get the number of topic assignments for word in the documents. For `phi`, the topic-term matrix, I aggregated by topic and word, counted the number of times each word was assigned to each topic, and then sorted the resulting dataframe alphabetically by word, so that it matches the order of the vocabulary frame. Here I used the `beta` hyperparamter as the smoothing value.

In [11]:
phi_df = df.groupby(['topic', 'type'])['type'].count().reset_index(name ='token_count')
phi_df = phi_df.sort_values(by='type', ascending=True)

phi_df[:10]

,topic,type,token_count
902220,200,aaa,2
220727,45,aaa,1
802489,177,aaa,10
484128,104,aaa,1
104943,19,aaa,5
930031,207,aaa,7
633014,138,aaa,3
715361,157,aaa,1
983211,219,aaa,4
35018,7,aaa,31


In [12]:
phi = pivot_and_smooth(phi_df, beta, 'topic', 'type', 'token_count')

# phi[:10]

Then we do this again, but focused on the documents and topics, to generate the `theta` document-topic matrix. Here I used `alpha` as the smoothing value.

In [13]:
theta_df = df.groupby(['#doc', 'topic'])['topic'].count().reset_index(name ='topic_count')

theta_df[:10]

,#doc,topic,topic_count
0,0,11,10
1,0,14,7
2,0,94,14
3,0,102,2
4,0,113,23
5,0,136,12
6,0,144,1
7,0,165,1
8,0,171,12
9,0,175,1


In [14]:
theta = pivot_and_smooth(theta_df, alpha , '#doc', 'topic', 'topic_count')

# theta[:10]

These data processing steps, as always, represent about 90% of the work needed to use the LDAvis library with a MALLET topic model. Now that I have all of the data in place, I can queue that data up and pass it to the visualization library. 

In [15]:
import pyLDAvis

data = {'topic_term_dists': phi, 
        'doc_topic_dists': theta,
        'doc_lengths': list(docs['doc_length']),
        'vocab': list(vocab['type']),
        'term_frequency': list(vocab['term_freq'])
       }

I used the code examples I mentioned earlier for setting the keys of the data dictionary. Keeping the data in sorted dataframes helps ensure that the order is consistent and preserved as it is moved into a list for analysis.

This data is then passed the visualization library, first for preparation and then for display. In preparing the data, the library computes the distances between topics and then projects those distances into a two-dimensional space using "multidimensional scaling" or principle component analysis. In the resulting visualization, the overlap in words of the topic is represented by shared location in space, and the greater the distance between topics, the larger the dissimilarity between the weight of the words that comprise the topic. The second element that the library computes is the most relevant terms for each topic with a sliding scale that either ranks on the overall frequency of the word (the default) or the distinctiveness of the word (lambda = 0.0 on the resulting slider). This provides a richer view of the topic assignments and is useful in labeling for distinguishing between topics.

In [16]:
vis_data = pyLDAvis.prepare(**data)

In [17]:
pyLDAvis.display(vis_data)

The resulting visualization indicates that this topic model has split the words of the corpus into topics of very similar size (all the circles are about the same size), with clusters of related topics. In addition, words such as "sabbath" and "law" occur in many of the topics, as seen when hovering of those words in the right hand column. This indicates to me that the corpus could have been modeled with fewer topics. For a more generalized and predictive use case, a model with fewer topics would likely be more effective. At the same time, however, religious language is both symbolic and repetitious, with similar language being used in multiple discourses for very different effects. The balance between too much specificity or over-fitting and over-generalization is made even more complex by this feature of the ways this community uses language.

Overall, however, the model and this visualization generate an interesting picture of the periodical literature of the SDA. The triangular outline of the graph indicates three dominant topical areas in the literature: theological concerns mapped in the lower left, denominational concerns in the lower right, and health concerns mapped to the center top. 

This representation of the topic model corresponds well enough to those I created using [Andrew Goldstone's topic model browser](https://agoldst.github.io/dfr-browser/) that I am relatively confident that the data is being processed correctly. The identification and use of the hyperparameter values is one of the elements of working with topic models that is still a bit murky to me, so I am happy to receive feedback on my data munging strategies (though I think with the latest corrections this is actually in the ballpark). Hopefully this provides a useful guide for taking data from MALLET and transforming it for visualization with LDAvis.  

In [18]:
vizDir = "/Users/jeriwieringa/Dissertation/site/files/interact/"
pyLDAvis.save_html(vis_data, os.path.join(vizDir, 'pyLDAviz.html'))